# World Happiness Report — Analyze Phase

Business question: **which factors are most strongly associated with national happiness, and
does that hold across regions?** Uses the cleaned panel from the Process phase
(`data/processed/happiness_panel.parquet`, 782 rows).

## Setup

In [1]:
import pandas as pd
import os

PROC = "../data/processed"
SUMMARY_DIR = "../data/summary"
os.makedirs(SUMMARY_DIR, exist_ok=True)

panel = pd.read_parquet(os.path.join(PROC, "happiness_panel.parquet"))
factors = ["gdp_per_capita", "social_support", "health_life_expectancy", "freedom",
           "corruption_perception", "generosity"]
print(f"Loaded panel: {panel.shape}")

Loaded panel: (782, 12)

## 1. Which factor correlates most with happiness?

In [2]:
corr = panel[factors + ["happiness_score"]].corr()["happiness_score"].drop("happiness_score").sort_values(ascending=False)
corr.to_csv(os.path.join(SUMMARY_DIR, "factor_correlations.csv"), header=["correlation"])
corr.round(3)

gdp_per_capita            0.789
health_life_expectancy    0.742
social_support            0.649
freedom                   0.551
corruption_perception     0.398
generosity                0.138
Name: happiness_score, dtype: float64

**GDP per capita and health/life expectancy are by far the strongest correlates. Generosity is by far the weakest** — despite being one of the report's 6 headline factors.

## 2. Regional averages (2019)

In [3]:
latest = panel[panel["year"] == 2019].copy()
region_avg = latest.groupby("region")["happiness_score"].agg(["mean", "count"]).round(2).sort_values("mean", ascending=False)
region_avg.to_csv(os.path.join(SUMMARY_DIR, "regional_averages_2019.csv"))
region_avg

                                   mean  count
region
Australia and New Zealand         7.27      2
North America                     7.08      2
Western Europe                    6.84     21
Latin America and Caribbean       5.95     21
Eastern Asia                      5.69      6
Central and Eastern Europe        5.56     29
Southeastern Asia                 5.27      9
Middle East and Northern Africa   5.24     19
Southern Asia                     4.53      7
Sub-Saharan Africa                4.29     39

## 3. Global trend, 2015-2019 (matched panel)

In [4]:
counts_per_country = panel.groupby("country")["year"].nunique()
stable_countries = counts_per_country[counts_per_country == 5].index
stable = panel[panel["country"].isin(stable_countries)]
trend = stable.groupby("year")["happiness_score"].mean().round(3)
trend.to_csv(os.path.join(SUMMARY_DIR, "global_trend.csv"), header=["mean_happiness_score"])
print(f"Using {len(stable_countries)} countries present in all 5 years for a fair like-for-like trend")
trend

Using 146 countries present in all 5 years for a fair like-for-like trend
year
2015    5.418
2016    5.411
2017    5.422
2018    5.453
2019    5.497
Name: happiness_score, dtype: float64

A modest but real upward trend — global average happiness among the same 146 countries rose from 5.42 (2015) to 5.50 (2019).

## 4. Top 10 / bottom 10 countries (2019)

In [5]:
top10 = latest.nsmallest(10, "rank")[["country", "region", "happiness_score", "rank"]]
bottom10 = latest.nlargest(10, "rank")[["country", "region", "happiness_score", "rank"]]
top10.to_csv(os.path.join(SUMMARY_DIR, "top10_2019.csv"), index=False)
bottom10.to_csv(os.path.join(SUMMARY_DIR, "bottom10_2019.csv"), index=False)
top10

     country                    region  happiness_score  rank
    Finland            Western Europe             7.769     1
    Denmark            Western Europe             7.600     2
     Norway            Western Europe             7.554     3
    Iceland            Western Europe             7.494     4
Netherlands            Western Europe             7.488     5
Switzerland            Western Europe             7.480     6
     Sweden            Western Europe             7.343     7
New Zealand Australia and New Zealand             7.307     8
     Canada             North America             7.278     9
    Austria            Western Europe             7.246    10

In [6]:
bottom10

                  country                          region  happiness_score  rank
             South Sudan             Sub-Saharan Africa             2.853   156
Central African Republic             Sub-Saharan Africa             3.083   155
             Afghanistan                  Southern Asia             3.203   154
                Tanzania             Sub-Saharan Africa             3.231   153
                  Rwanda             Sub-Saharan Africa             3.334   152
                   Yemen Middle East and Northern Africa             3.380   151
                  Malawi             Sub-Saharan Africa             3.410   150
                   Syria Middle East and Northern Africa             3.462   149
                Botswana             Sub-Saharan Africa             3.488   148
                   Haiti    Latin America and Caribbean             3.597   147

8 of the top 10 are Western European; 6 of the bottom 10 are Sub-Saharan African, alongside conflict-affected states (Afghanistan, Yemen, Syria).

## 5. What separates the top 10 from the bottom 10?

In [7]:
comp = pd.DataFrame({
    "top10_avg": latest.nsmallest(10, "rank")[factors].mean(),
    "bottom10_avg": latest.nlargest(10, "rank")[factors].mean(),
}).round(3)
comp["gap"] = (comp["top10_avg"] - comp["bottom10_avg"]).round(3)
comp = comp.sort_values("gap", ascending=False)
comp.to_csv(os.path.join(SUMMARY_DIR, "top_vs_bottom_factors.csv"))
comp

                         top10_avg  bottom10_avg    gap
gdp_per_capita               1.387         0.398  0.989
social_support               1.544         0.662  0.882
health_life_expectancy       1.018         0.426  0.592
freedom                      0.579         0.229  0.350
corruption_perception        0.319         0.123  0.196
generosity                   0.274         0.219  0.055

The gap between the happiest and least-happy countries is largest for **GDP per capita** and
**social support**, smallest for **generosity** — consistent with the correlation ranking in
step 1.

## Summary of key findings

1. **GDP per capita (r=0.79) and health/life expectancy (r=0.74) are the strongest correlates of
   happiness**; generosity (r=0.14) is the weakest by a wide margin.
2. **Regional gap is large**: Australia/NZ and North America average ~7.1-7.3; Sub-Saharan Africa
   averages 4.29 — a gap of roughly 3 points on the 0-10 scale.
3. **Global happiness rose modestly but consistently** from 2015 to 2019 (5.42 → 5.50) among the
   146 countries present every year.
4. **The top/bottom 10 gap mirrors the correlation ranking**: GDP and social support show the
   largest gaps between the happiest and least-happy countries; generosity shows almost none.

**Overall**: economic security and material social support — not generosity or even freedom —
are what most separates the happiest countries from the least happy in this data. This directly
shapes the Act-phase recommendations for where a development program should focus.